In [11]:
import torch
def gaussian_kernel_2d(sigma, kernel_size=6):
    if kernel_size is None:
        raise ValueError("No kernel provided")
    ax = torch.arange(kernel_size, dtype=torch.float32) - (kernel_size - 1) / 2.0
    x, y = torch.meshgrid(ax, ax, indexing='ij')
    print(x)
    k = torch.exp(-(x ** 2 + y ** 2) / (2 * sigma ** 2))
    k = k / k.sum()
    return k[None, None, :, :]
k=gaussian_kernel_2d(sigma=0.5)


tensor([[-2.5000, -2.5000, -2.5000, -2.5000, -2.5000, -2.5000],
        [-1.5000, -1.5000, -1.5000, -1.5000, -1.5000, -1.5000],
        [-0.5000, -0.5000, -0.5000, -0.5000, -0.5000, -0.5000],
        [ 0.5000,  0.5000,  0.5000,  0.5000,  0.5000,  0.5000],
        [ 1.5000,  1.5000,  1.5000,  1.5000,  1.5000,  1.5000],
        [ 2.5000,  2.5000,  2.5000,  2.5000,  2.5000,  2.5000]])


In [10]:
kernel_size=6
torch.arange(kernel_size, dtype=torch.float32)

tensor([0., 1., 2., 3., 4., 5.])

In [1]:
for sigma in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
    kernel_size = int(4 * sigma + 1) | 1
    print(kernel_size)

1
1
3
3
3
3
3
5
5


In [ ]:

def gaussian_blur_torch(img_2d, sigma):
    if sigma <= 0:
        return img_2d
    kernel = gaussian_kernel_2d(sigma)
    kernel = kernel.to(img_2d.device)
    padding = kernel.shape[-1] // 2
    img_4d = img_2d[None, None, :, :]
    return F.conv2d(img_4d, kernel, padding=padding)[0, 0]


def img_to_encoder_torch(img):
    return torch.flip(img.T, dims=[-1])


In [ ]:
def fit(self, optotype, blur_sigma=1.5):
    if optotype.dim() == 3:
        optotype = optotype.squeeze(0)
    h, _ = optotype.shape
    self.half_n = h // 2
    optotype_dev = optotype.to(self.device)
    blurred = gaussian_blur_torch(optotype_dev, blur_sigma)
    self.optotype_display = blurred.cpu().numpy()
    self.optotype_np = np.fliplr(blurred.cpu().numpy().T)
    self.optotype_torch = img_to_encoder_torch(blurred)